# VaR Estimation Models

Demonstrates all VaR estimation methods available in `risk-backtest`:
- Historical Simulation
- Normal (Parametric)
- Cornish-Fisher Expansion
- Extreme Value Theory (Peaks Over Threshold)
- GARCH Family (GARCH, GJR-GARCH, EGARCH, APARCH)
- Recursive Variance Calculation

In [ ]:
import numpy as np
import pandas as pd
from risk_backtest import (
    cornish_fisher_var,
    evt_var,
    garch_var,
    estimate_var,
    recursive_garch_variance,
    GARCH_MODELS,
)

print(f"Available GARCH models: {GARCH_MODELS}")

## Generate Realistic Return Data

In [ ]:
# Simulate returns with volatility clustering (GARCH-like)
rng = np.random.default_rng(123)
n = 1000

# GARCH(1,1) DGP
omega, alpha, beta = 0.00001, 0.08, 0.90
sigma2 = np.zeros(n)
returns = np.zeros(n)
sigma2[0] = omega / (1 - alpha - beta)

for t in range(1, n):
    sigma2[t] = omega + alpha * returns[t-1]**2 + beta * sigma2[t-1]
    returns[t] = np.sqrt(sigma2[t]) * rng.standard_t(df=5)

print(f"Generated {n} returns with vol clustering")
print(f"  Mean: {returns.mean():.6f}")
print(f"  Std:  {returns.std():.6f}")
print(f"  Skew: {pd.Series(returns).skew():.4f}")
print(f"  Kurt: {pd.Series(returns).kurtosis():.4f} (excess)")

## Compare All Methods (with verbose output)

In [ ]:
# Quick comparison of non-GARCH methods
results = estimate_var(
    returns,
    confidence_level=0.99,
    methods=["historical", "normal", "cornish_fisher", "evt"],
    verbose=True,
)

In [ ]:
# Summary table
summary = []
for method, res in results.items():
    val = res.var if hasattr(res, 'var') else res
    summary.append({"Method": method, "VaR (99%)": f"{val:.4%}"})

pd.DataFrame(summary)

## Cornish-Fisher Expansion

In [ ]:
# Cornish-Fisher adjusts for skewness and kurtosis
var_cf = cornish_fisher_var(returns, confidence_level=0.99, verbose=True)
print(f"\n  99% VaR = {var_cf:.4%}")

# Compare at different confidence levels
for cl in [0.95, 0.975, 0.99, 0.995]:
    v = cornish_fisher_var(returns, confidence_level=cl)
    print(f"  {cl*100:.1f}% VaR = {v:.4%}")

## Extreme Value Theory (Peaks Over Threshold)

In [ ]:
# EVT fits a GPD to tail exceedances
evt_result = evt_var(
    returns,
    confidence_level=0.99,
    threshold_quantile=0.90,  # Use top 10% of losses
    verbose=True,
)

print(f"\n  VaR (99%):    {evt_result.var:.4%}")
print(f"  ES / CVaR:    {evt_result.es:.4%}")
print(f"  GPD shape ξ:  {evt_result.shape:.4f}")
print(f"  GPD scale β:  {evt_result.scale:.6f}")

In [ ]:
# Sensitivity to threshold choice
print("Threshold quantile sensitivity:")
print(f"{'Quantile':<12} {'N_Exceed':<10} {'VaR':<10} {'ES':<10} {'ξ':<8}")
for q in [0.80, 0.85, 0.90, 0.93, 0.95]:
    r = evt_var(returns, threshold_quantile=q)
    print(f"{q:<12.2f} {r.n_exceedances:<10} {r.var:<10.4%} {r.es:<10.4%} {r.shape:<8.4f}")

## GARCH Family Models

All four GARCH variants using Maximum Likelihood Estimation (requires `arch` package).

In [ ]:
# Standard GARCH(1,1)
result_garch = garch_var(
    returns,
    confidence_level=0.99,
    model_type="garch",
    dist="t",  # Student-t innovations
    verbose=True,
)

In [ ]:
# GJR-GARCH (captures leverage effect)
result_gjr = garch_var(
    returns,
    confidence_level=0.99,
    model_type="gjr-garch",
    dist="t",
    verbose=True,
)
print(f"\n  Leverage (γ): {result_gjr.extra_params['gamma']:.4f}")
print(f"  → Negative shocks increase vol {result_gjr.extra_params['gamma']*100:.1f}% more")

In [ ]:
# EGARCH (exponential — no positivity constraints needed)
result_egarch = garch_var(
    returns,
    confidence_level=0.99,
    model_type="egarch",
    dist="t",
    verbose=True,
)

In [ ]:
# APARCH (asymmetric power — flexible power parameter δ)
result_aparch = garch_var(
    returns,
    confidence_level=0.99,
    model_type="aparch",
    dist="t",
    verbose=True,
)
print(f"\n  Power (δ): {result_aparch.extra_params['delta']:.4f}")
print(f"  (δ=2 → standard GARCH, δ=1 → absolute value model)")

In [ ]:
# Compare all GARCH models
garch_comparison = pd.DataFrame([
    {"Model": r.model_type.upper(), "VaR": f"{r.var:.4%}",
     "Cond. Vol": f"{r.conditional_vol:.4%}", "Persistence": f"{r.persistence:.4f}",
     "LogLik": f"{r.log_likelihood:.1f}"}
    for r in [result_garch, result_gjr, result_egarch, result_aparch]
])
garch_comparison

## Recursive Variance Calculation (Lightweight, No Dependencies)

Use fitted parameters to compute the full conditional variance path in O(n) time.
No `arch` package needed — just NumPy.

In [ ]:
# Use parameters from the fitted GARCH model
sigma2_garch = recursive_garch_variance(
    returns,
    omega=result_garch.omega,
    alpha=result_garch.alpha,
    beta=result_garch.beta,
    model_type="garch",
    verbose=True,
)

daily_vol = np.sqrt(sigma2_garch)
print(f"\n  Conditional vol series: {len(daily_vol)} points")
print(f"  Can be used as rolling VaR input for backtesting!")

In [ ]:
# GJR-GARCH recursive (captures asymmetry without re-fitting)
sigma2_gjr = recursive_garch_variance(
    returns,
    omega=result_gjr.omega,
    alpha=result_gjr.alpha,
    beta=result_gjr.beta,
    gamma=result_gjr.extra_params['gamma'],
    model_type="gjr-garch",
    verbose=True,
)

In [ ]:
# Plot conditional volatilities (if matplotlib available)
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(np.sqrt(sigma2_garch) * 100, label='GARCH', alpha=0.8)
    ax.plot(np.sqrt(sigma2_gjr) * 100, label='GJR-GARCH', alpha=0.8)
    ax.set_ylabel('Daily Volatility (%)')
    ax.set_xlabel('Observation')
    ax.set_title('Recursive Conditional Volatility')
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for charts: pip install matplotlib")

## Full Comparison: All Methods

In [ ]:
# Compare everything in one call
all_results = estimate_var(
    returns,
    confidence_level=0.99,
    methods=["historical", "normal", "cornish_fisher", "evt",
             "garch", "gjr-garch", "egarch", "aparch"],
    garch_dist="t",
)

# Build comparison table
rows = []
for method, res in all_results.items():
    val = res.var if hasattr(res, 'var') else res
    rows.append({"Method": method, "VaR (99%)": val})

df_compare = pd.DataFrame(rows).sort_values("VaR (99%)", ascending=False)
df_compare["VaR (99%)"] = df_compare["VaR (99%)"].apply(lambda x: f"{x:.4%}")
df_compare.reset_index(drop=True)